In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Creează un plugin Transpiler

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Îți recomandăm să folosești aceste versiuni sau unele mai noi.

```
qiskit[all]~=2.3.0
```
</details>
Crearea unui [plugin Transpiler](transpiler-plugins) este o modalitate excelentă de a-ți împărtăși codul de transpilare cu comunitatea Qiskit, permițând altor utilizatori să beneficieze de funcționalitatea pe care ai dezvoltat-o. Îți mulțumim pentru interesul de a contribui la comunitatea Qiskit!

Înainte de a crea un plugin Transpiler, trebuie să decizi ce tip de plugin este potrivit pentru situația ta. Există trei tipuri de plugin-uri Transpiler:
- [**Plugin de etapă Transpiler**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins). Alege-l dacă definești un manager de pași care poate înlocui una dintre cele [6 etape](transpiler-stages) ale unui manager de pași stadializat prestabilit.
- [**Plugin de sinteză unitară**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin). Alege-l dacă codul tău de transpilare primește ca intrare o matrice unitară (reprezentată ca un array Numpy) și produce o descriere a unui Circuit cuantic care implementează acea unitară.
- [**Plugin de sinteză de nivel înalt**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin). Alege-l dacă codul tău de transpilare primește ca intrare un „obiect de nivel înalt", cum ar fi un operator Clifford sau o funcție liniară, și produce o descriere a unui Circuit cuantic care implementează acel obiect de nivel înalt. Obiectele de nivel înalt sunt reprezentate de subclase ale clasei [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

Odată ce ai stabilit ce tip de plugin să creezi, urmează acești pași pentru a crea plugin-ul:

1. Creează o subclasă a clasei abstracte de plugin corespunzătoare:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) pentru un plugin de etapă Transpiler,
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) pentru un plugin de sinteză unitară, și
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) pentru un plugin de sinteză de nivel înalt.
2. Expune clasa ca un [entry point setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) în metadatele pachetului, de obicei prin editarea fișierului `pyproject.toml`, `setup.cfg` sau `setup.py` al pachetului tău Python.

Nu există o limită pentru numărul de plugin-uri pe care un singur pachet le poate defini, dar fiecare plugin trebuie să aibă un nume unic. SDK-ul Qiskit în sine include un număr de plugin-uri, ale căror nume sunt rezervate. Numele rezervate sunt:

- Plugin-uri de etapă Transpiler: Vezi [acest tabel](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- Plugin-uri de sinteză unitară: `default`, `aqc`, `sk`
- Plugin-uri de sinteză de nivel înalt:

| Clasa Operation | Numele Operation | Nume rezervate |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

În secțiunile următoare, prezentăm exemple ale acestor pași pentru diferitele tipuri de plugin-uri. În aceste exemple, presupunem că creăm un pachet Python numit `my_qiskit_plugin`. Pentru informații despre crearea pachetelor Python, poți consulta [acest tutorial](https://packaging.python.org/en/latest/tutorials/packaging-projects/) de pe site-ul Python.
## Exemplu: Creează un plugin de etapă Transpiler
În acest exemplu, creăm un plugin de etapă Transpiler pentru etapa `layout` (vezi [Etapele Transpiler](transpiler-stages) pentru o descriere a celor 6 etape ale pipeline-ului de transpilare integrat al Qiskit).
Plugin-ul nostru rulează pur și simplu [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) pentru un număr de încercări care depinde de nivelul de optimizare solicitat.

Mai întâi, creăm o subclasă a [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). Există o metodă pe care trebuie să o implementăm, numită [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). Această metodă primește ca intrare un [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) și returnează managerul de pași pe care îl definim. Obiectul PassManagerConfig stochează informații despre Backend-ul țintă, cum ar fi harta de cuplare și Gate-urile de bază ale acestuia.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

Acum, expunem plugin-ul adăugând un entry point în metadatele pachetului nostru Python.
Aici, presupunem că clasa pe care am definit-o este expusă într-un modul numit `my_qiskit_plugin`, de exemplu prin import în fișierul `__init__.py` al modulului `my_qiskit_plugin`.
Edităm fișierul `pyproject.toml`, `setup.cfg` sau `setup.py` al pachetului nostru (în funcție de tipul de fișier pe care l-ai ales pentru a stoca metadatele proiectului tău Python):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>
Vezi [tabelul etapelor plugin-ului Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) pentru entry point-urile și așteptările pentru fiecare etapă Transpiler.

Pentru a verifica că plugin-ul tău este detectat cu succes de Qiskit, instalează pachetul plugin-ului tău și urmează instrucțiunile de la [Plugin-uri Transpiler](transpiler-plugins#list-available-transpiler-stage-plugins) pentru listarea plugin-urilor instalate și asigură-te că plugin-ul tău apare în listă:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

Dacă plugin-ul nostru exemplu ar fi instalat, atunci numele `my_layout` ar apărea în această listă.

Dacă vrei să folosești o etapă Transpiler integrată ca punct de plecare pentru plugin-ul tău de etapă Transpiler, poți obține managerul de pași pentru o etapă Transpiler integrată folosind [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). Următoarea celulă de cod arată cum să faci asta pentru a obține etapa de optimizare integrată pentru nivelul de optimizare 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## Exemplu: Creează un plugin de sinteză unitară
În acest exemplu, vom crea un plugin de sinteză unitară care folosește pur și simplu pasul de transpilare built-in [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) pentru a sintetiza un Gate. Desigur, propriul tău plugin va face ceva mai interesant decât atât.

Clasa [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) definește interfața și contractul pentru
plugin-urile de sinteză unitară. Metoda principală este
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run),
care primește ca intrare un array Numpy ce stochează o matrice unitară
și returnează un [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) reprezentând Circuit-ul sintetizat din acea matrice unitară.
Pe lângă metoda `run`, există o serie de metode de tip proprietate care trebuie definite.
Vezi [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) pentru documentația tuturor proprietăților obligatorii.

Să creăm subclasa noastră UnitarySynthesisPlugin:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

Dacă descoperi că intrările disponibile pentru metoda [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
sunt insuficiente pentru scopurile tale, te rugăm să [deschizi un issue](https://github.com/Qiskit/qiskit/issues/new/choose) explicând cerințele tale. Modificările aduse interfeței plugin-ului, cum ar fi adăugarea unor intrări opționale suplimentare, vor fi realizate într-un mod compatibil cu versiunile anterioare, astfel încât să nu necesite modificări din partea plugin-urilor existente.

> **Note:** Toate metodele cu prefixul `supports_` sunt rezervate pe o clasă derivată din `UnitarySynthesisPlugin` ca parte a interfeței. Nu trebuie să definești metode personalizate `supports_*` pe o subclasă care nu sunt definite în clasa abstractă.

Acum, expunem plugin-ul adăugând un entry point în metadatele pachetului nostru Python.
Presupunem că clasa pe care am definit-o este expusă într-un modul numit `my_qiskit_plugin`, de exemplu prin importare în fișierul `__init__.py` al modulului `my_qiskit_plugin`.
Edităm fișierul `pyproject.toml`, `setup.cfg` sau `setup.py` al pachetului nostru:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

Ca și înainte, dacă proiectul tău folosește `setup.cfg` sau `setup.py` în loc de `pyproject.toml`, consultă [documentația setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) pentru a adapta aceste linii situației tale.

Pentru a verifica că plugin-ul tău este detectat cu succes de Qiskit, instalează pachetul plugin-ului și urmează instrucțiunile din [Transpiler plugins](transpiler-plugins#list-available-unitary-synthesis-plugins) pentru a lista plugin-urile instalate și asigură-te că plugin-ul tău apare în listă:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

Dacă plugin-ul nostru exemplu ar fi instalat, atunci numele `my_unitary_synthesis` ar apărea în această listă.

Pentru a acomoda plugin-urile de sinteză unitară care expun mai multe opțiuni,
interfața plugin-ului are o opțiune pentru ca utilizatorii să furnizeze un
dicționar de configurare liber. Acesta va fi transmis metodei `run`
prin argumentul keyword `options`. Dacă plugin-ul tău are aceste opțiuni de configurare, ar trebui să le documentezi clar.

## Exemplu: Creează un plugin de sinteză de nivel înalt

În acest exemplu, vom crea un plugin de sinteză de nivel înalt care folosește pur și simplu funcția built-in [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) pentru a sintetiza un operator Clifford.

Clasa [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) definește interfața și contractul pentru plugin-urile de sinteză de nivel înalt. Metoda principală este [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
Argumentul pozițional `high_level_object` este o [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) care reprezintă obiectul „de nivel înalt" ce urmează să fie sintetizat. De exemplu, poate fi o
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) sau un
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
Sunt prezenți următorii argumenți cu cuvânt cheie:
- `target` specifică Backend-ul țintă, permițând plugin-ului
să acceseze toate informațiile specifice țintei,
precum harta de cuplare, setul de Gate-uri suportate și altele
- `coupling_map` specifică doar harta de cuplare și este folosit numai când `target` nu este specificat.
- `qubits` specifică lista de Qubiți peste care este definit
obiectul de nivel înalt, în cazul în care sinteza se face pe Circuit-ul fizic.
O valoare de ``None`` indică faptul că layout-ul nu a fost încă ales și că Qubiții fizici din țintă sau harta de cuplare pe care operează această operație nu au fost încă determinați.
- `options`, un dicționar de configurare liber pentru opțiunile specifice plugin-ului. Dacă plugin-ul tău are aceste opțiuni de configurare, ar trebui să le documentezi clar.

Metoda `run` returnează un [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
care reprezintă Circuit-ul sintetizat din acel obiect de nivel înalt.
Este, de asemenea, permis să returneze `None`, indicând că plugin-ul nu poate sintetiza obiectul de nivel înalt dat.
Sinteza efectivă a obiectelor de nivel înalt este realizată de pasul Transpiler
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

Pe lângă metoda `run`, există o serie de metode de proprietate care trebuie definite.
Vezi [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) pentru documentația tuturor proprietăților obligatorii.

Hai să definim subclasa noastră HighLevelSynthesisPlugin: